# 🚗 Car Price Prediction — XGBoost
**Dataset:** Car_details_v3.csv | **Algorithm:** XGBoost Regressor

> Run each cell in order. Upload your CSV when prompted.

## 📦 Step 1 — Install & Import Libraries

In [ ]:
# XGBoost is pre-installed in Colab — no extra install needed
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
print('✅ Libraries loaded')

## 📁 Step 2 — Upload Dataset

In [ ]:
from google.colab import files
uploaded = files.upload()  # Upload Car_details_v3.csv
df = pd.read_csv('Car_details_v3.csv')
print(f'✅ Loaded: {df.shape[0]} rows, {df.shape[1]} columns')
df.head()

## 🧹 Step 3 — Data Cleaning

In [ ]:
# Strip units from string columns
def extract_number(series):
    return pd.to_numeric(series.str.extract(r'([\d.]+)')[0], errors='coerce')

df['mileage_num']   = extract_number(df['mileage'])
df['engine_num']    = extract_number(df['engine'])
df['max_power_num'] = extract_number(df['max_power'])

# Drop original string cols and torque (too complex to parse)
df.drop(columns=['name', 'mileage', 'engine', 'max_power', 'torque'], inplace=True)

# Drop missing values
df.dropna(inplace=True)

print(f'✅ Clean dataset: {df.shape[0]} rows')
df.info()

## 🔠 Step 4 — Encode Categorical Features

In [ ]:
cat_cols = ['fuel', 'seller_type', 'transmission', 'owner']
le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col])

print('✅ Encoding done')
print('Fuel      : 0=CNG, 1=Diesel, 2=LPG, 3=Petrol')
print('Seller    : 0=Dealer, 1=Individual, 2=Trustmark')
print('Transmit  : 0=Automatic, 1=Manual')
print('Owner     : 0=First, 1=Fourth & Above, 2=Second, 3=Test Drive, 4=Third')
df.head()

## ✂️ Step 5 — Train / Test Split

In [ ]:
X = df.drop(columns=['selling_price'])
y = df['selling_price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'✅ Train: {len(X_train)} rows | Test: {len(X_test)} rows')
print(f'Features: {X.columns.tolist()}')

## 🤖 Step 6 — Train XGBoost Model

In [ ]:
model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=0
)

model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
print('✅ Model trained!')

## 📊 Step 7 — Evaluate Model

In [ ]:
y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print('=' * 40)
print(f'  RMSE : ₹{rmse:,.0f}')
print(f'  R²   : {r2:.4f}  ({r2*100:.1f}% variance explained)')
print('=' * 40)

## 📈 Step 8 — Feature Importance Plot

In [ ]:
importance = pd.Series(model.feature_importances_, index=X.columns).sort_values()

plt.figure(figsize=(8, 5))
plt.barh(importance.index, importance.values, color='#378ADD')
plt.xlabel('Importance Score')
plt.title('XGBoost — Feature Importance')
plt.tight_layout()
plt.show()

## 🔍 Step 9 — Actual vs Predicted Plot

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred, alpha=0.3, color='#378ADD', edgecolors='none', s=15)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=1.5, label='Perfect fit')
plt.xlabel('Actual Price (₹)')
plt.ylabel('Predicted Price (₹)')
plt.title('Actual vs Predicted Car Price')
plt.legend()
plt.tight_layout()
plt.show()

## 🚘 Step 10 — Predict Your Own Car

In [ ]:
# Change these values to predict any car's price
custom_car = pd.DataFrame([{
    'year':          2018,
    'km_driven':     45000,
    'fuel':          1,       # 1 = Diesel
    'seller_type':   1,       # 1 = Individual
    'transmission':  1,       # 1 = Manual
    'owner':         0,       # 0 = First Owner
    'mileage_num':   20.0,    # kmpl
    'engine_num':    1500.0,  # CC
    'max_power_num': 100.0,   # bhp
    'seats':         5.0
}])

price = model.predict(custom_car)[0]
print(f'🚗 Estimated Selling Price: ₹{price:,.0f}')